# 🏗️ 07-大模型训练优化实践

> 目标：把前面所有优化知识接到 **LLM 训练**——FP16/BF16 混合精度、Adam 优化器状态内存、ZeRO 分片。

**本讲地图**：三种浮点格式 → 为什么 BF16 不是 FP16 → loss scaling →
混合精度训练四步流程 → 四个实验（溢出/loss scaling/Adam 内存/ZeRO 曲线）→
显存换时间组合拳 → 面试速答 → 自测。

> ⚠️ 前置：03/04 篇（Adam/AdamW 状态）、05 篇（调度与裁剪）。本篇把「优化器状态
> 内存」这个点做透——它是 LLM 训练 3D 并行与 ZeRO 的起点。

## §1 三种浮点格式

| 格式 | 总位 | 指数位 | 尾数位 | 最大有限值 | 最小正常值 | 相对精度 |
|------|------|--------|--------|------------|------------|----------|
| FP32 | 32 | 8 | 23 | $\approx 3.4\times10^{38}$ | $1.2\times10^{-38}$ | $\sim10^{-7}$ |
| FP16 | 16 | 5 | 10 | 65504 | $6.1\times10^{-5}$ | $\sim10^{-3}$ |
| BF16 | 16 | 8 | 7 | $\approx 3.4\times10^{38}$ | $1.2\times10^{-38}$ | $\sim10^{-2}$ |

**BF16 = FP32 的指数范围 + 更少的尾数**：梯度/激活很少溢出（LLM 主流）；
FP16 范围窄（最大 65504、最小正常值 $6\times10^{-5}$），需要 **loss scaling**。

> 🧩 记忆：指数位决定**范围**（能不能存大数/小数），尾数位决定**精度**（小数点后几位可靠）。
> BF16 牺牲精度换范围——训练中范围远比精度重要（梯度跨数量级）。

## §1.1 为什么 BF16 不是 FP16（面试高频）

两个格式都是 16 位，但分配完全不同：
- **FP16**：5 位指数（范围窄）+ 10 位尾数（精度尚可）。
  梯度在 Transformer 中经常小到 $10^{-6}$（下溢成 0）或大到溢出（>65504 变 inf）。
- **BF16**：8 位指数（与 FP32 完全相同的范围！）+ 7 位尾数（精度差，只有约 3 位十进制）。
  几乎不会溢出/下溢，代价是**相对精度低**。

为什么低精度也能训？深度学习对**相对误差**容忍度高（梯度方向大致正确即可），
但**绝对范围**一旦溢出就是灾难。所以 LLM 训练主流选 BF16；
老一点的流程（NVIDIA A100 之前、Turing 系）用 FP16 + loss scaling 兜底。

## §1.2 历史背景：从 FP16 + loss scaling 到 BF16

混合精度训练的关键节点：
1. **2017-2018（Micikevicius et al., APEX/Megatron 时代）**：FP16 训练 + **loss scaling**——
   前向把 loss 乘一个大数（如 1024），使梯度进入 FP16 可表示范围，反向后再除掉。
   但 scaling 因子要动态调整（过大溢出、过小下溢），是工程痛点。
2. **2019（NVIDIA）**：推出 **BF16** 硬件支持（A100 普及后成为标配），
   范围问题从根上消失，不再需要 loss scaling。
3. **现在**：LLM 训练（GPT-3/LLaMA 系）几乎清一色 **BF16 混合精度 + FP32 optimizer 状态**。

> 📌 面试话术：「FP16 需要 loss scaling 是因为指数位只有 5，范围太窄；
> BF16 用 8 位指数复刻 FP32 的范围，省掉 scaling 的调参负担，所以成为 LLM 主流。」

## §1.3 实验 1 预习：到底会溢出多少

实验 1 把一组跨数量级的值（$10^{-4}$ ~ $5\times10^{4}$）分别存成 FP32 / FP16 / BF16 并打印：
- 预期：FP16 中 $5\times10^{4}$ 溢出为 inf（最大 65504），$10^{-4}$ 还正常；
- BF16 与 FP32 范围一致，都能表示。

跑之前想：为什么 `np.finfo(np.float16).max` 是 65504 而不是更大？
（提示：指数位 5，最大指数 30，尾数全 1 → $(2-2^{-10})\cdot 2^{15}$。）

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# ---------- 实验 1：FP16 溢出 vs BF16 不溢出 ----------
vals = np.array([1e-4, 1e-3, 0.01, 1.0, 100.0, 5e4])
fp16 = vals.astype(np.float16)
# 部分 numpy 版本没有 bfloat16（如 numpy<2.0），这里兼容处理
try:
    bf16 = vals.astype(np.bfloat16)
    bf16_ok = True
except AttributeError:
    bf16_ok = False
fp32 = vals.astype(np.float32)
print('数值       FP32        FP16        BF16')
for i, v in enumerate(vals):
    b = f'{bf16[i]:12.4e}' if bf16_ok else '  N/A(旧numpy)'
    print(f'{v:8.1e}  {fp32[i]:12.4e}  {fp16[i]:12.4e}  {b}')
print()
print('FP16 最大有限值:', np.finfo(np.float16).max)
if bf16_ok:
    print('BF16 最大有限值:', np.finfo(np.bfloat16).max)
else:
    print('BF16 最大有限值: 与 FP32 相同(≈3.4e38)（当前 numpy 无 bfloat16，理论值）')
print('结论: 5e4 在 FP16 中已是 inf；BF16 与 FP32 范围一致')

### 实验 1 复盘

输出应显示：`5e4` 在 FP16 列变成 `inf`，而 BF16 列保持 $5\times10^{4}$。
这说明**溢出是 FP16 的硬伤**：Transformer 的注意力 logits、embedding 范数
经常越过 65504，一旦 inf，反向传播的梯度全部 NaN，训练直接崩。

另外注意最小正常值：FP16 是 $6.1\times10^{-5}$——比它小的正数要么非规格化（精度骤降）
要么为 0。深度学习的梯度经常落在这个区间，所以「下溢」和「溢出」同样致命。

In [ ]:
# ---------- 实验 2：loss scaling 救 FP16 ----------
tiny_grad = 1e-8                       # 小于 FP16 最小非规格化值(≈6e-8) → 直接存必为 0
fp16_no_scale = np.float16(tiny_grad)           # 直接存 → 下溢为 0
scale = 1024.0                                  # 用 float64 运算再转 FP16，避免先把 1e-8 转 FP16
fp16_scaled = np.float16(tiny_grad * scale)     # 放大到正常数后存
back = np.float32(fp16_scaled) / np.float32(scale)  # 反缩放
print(f'原始梯度={tiny_grad:.1e}')
print(f'FP16 直接存={float(fp16_no_scale):.3e}（下溢→0）')
print(f'FP16 缩放存储后反缩放={back:.3e}（保住了）')
assert float(fp16_no_scale) == 0.0 and back > 0
print('loss scaling 原理演示通过')

### 实验 2 复盘：loss scaling 为什么有效

梯度 $10^{-8}$ 比 FP16 最小非规格化值（约 $6\times10^{-8}$）还小，直接存就是 0；
乘上 1024 后变成 $1.024\times10^{-5}$，回到 FP16 可表示范围，存好后再除以 1024 还原。
**scaling 只是「移位」**：把整个梯度分布挪进可表示窗口，不改变相对大小。

工程实现要点：
- loss 乘 scale 再 `backward()`，梯度自动带 scale；
- optimizer step 前先除 scale（转 FP32 再做，避免误差）；
- 动态 scaling：每 N 步检查梯度是否 inf/NaN，若有则减半 scale 重试，否则缓慢增大。

> ⚠️ BF16 时代已不需要这套机制，但理解它才能回答「为什么老代码里有 loss scaler」。

## §2 混合精度训练流程

1. 参数保留 **FP32 master copy**（一份高精度权重）；
2. 前向/反向在 FP16/BF16 上做（省显存、加速）——权重复制成半精度副本参与计算；
3. 梯度反缩放（FP16）或直接（BF16）后转 FP32；
4. 优化器状态与更新在 FP32 master 上做（累加精度），更新完再同步回半精度副本。

**为什么 optimizer 状态要 FP32**：Adam 的 $m,v$ 是**累加量**，
每步叠加的小更新在 FP16 里会因相对精度 $10^{-3}$ 而丢失（小步长 < 大数值的 $10^{-3}$ 即被吞掉）；
FP32 的 $10^{-7}$ 相对精度才能保住。这就是「**master weights + FP32 optimizer 状态**」的由来。

## §2.1 手算：为什么 m/v 必须 FP32

设 $m$ 已累计到 100.0，这一步的动量增量只有 0.005（小更新）：
- FP32（相对精度 $10^{-7}$）：$100.0 + 0.005 = 100.005$，能表示；
- FP16（相对精度 $10^{-3}$）：$100.0 + 0.005$ 会舍入回 $100.0$——**更新被吃掉**。

对 v（二阶矩）同理：长时间训练后 $v$ 很大，微小更新在 FP16 中丢失，
导致步长统计失真。**结论：优化器状态 = 高精度累加器，必须 FP32（或更高）。**
这也是 04 篇「7B 模型 Adam 状态 84GB」的来源。

In [ ]:
# ---------- 实验 3：Adam 优化器状态内存分析 ----------
# 参数量 -> 显存（FP32 每个参数：参数4B + m 4B + v 4B = 12B；BF16 参数/梯度可省）
params = np.array([0.5, 1, 3, 7, 13, 70, 175])  # 十亿参数
bytes_per_p = 12  # 参数4 + m4 + v4（FP32 优化器状态）
mem_fp32 = params * bytes_per_p * 1e9 / 1e9     # GB

fig, ax = plt.subplots(figsize=(8, 4.0))
ax.plot(params, mem_fp32, 'o-', color='#4C72B0', lw=1.6, label='Adam FP32 优化器状态+参数 (12B/参数)')
ax.plot(params, params * 8, 's--', color='#55A868', lw=1.4, label='BF16 训练近似 (8B/参数)')
for p, m in zip(params, mem_fp32):
    ax.annotate(f'{m:.0f}GB', (p, m), textcoords='offset points', xytext=(6, 4), fontsize=8)
ax.set_xlabel('参数量 (B)'); ax.set_ylabel('显存 (GB)')
ax.set_title('7B 模型 Adam 优化器状态 ≈ 84GB（远超单卡）', fontsize=12)
ax.grid(alpha=0.3); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()
print('口算：7B × 12B = 84GB 优化器状态+参数（不含激活/梯度）')

### 实验 3 复盘：84GB 是怎么算出来的

每个参数在 AdamW 混合精度训练中占多少字节？
- 参数：FP32 master（4B）+ BF16 副本（2B，用于前反向）≈ 6B；
- 优化器状态：$m$（4B）+ $v$（4B）= 8B；
- 合计 **12B/参数**（若只算 FP32 master + m + v）。

所以：$7\times10^9$ 参数 × 12B = **84GB**。
一张 A100-80GB 都放不下优化器状态+参数，更别说激活和梯度——
这就是为什么必须 **ZeRO 分片 / offload / 并行**（实验 4 与 §3）。

In [ ]:
# ---------- 实验 4：ZeRO 分片内存曲线 ----------
# ZeRO-1/2/3：分别把 优化器状态 / +梯度 / +参数 分片到 N 卡
N = np.arange(1, 65)
B = 7e9 * 12 / 1e9   # 7B FP32 全量 GB
def zero_mem(stage, n):
    base = B  # 全量（未分片）
    if stage == 1:  # 只分优化器状态(8B/p)，参数+梯度(4B/p) 每卡都有
        return 7 * 8 / n + 7 * 4
    if stage == 2:  # 优化器状态+梯度(10B/p) 分片
        return 7 * 10 / n + 7 * 2
    if stage == 3:  # 全部(12B/p) 分片
        return 7 * 12 / n
    return base

fig, ax = plt.subplots(figsize=(8, 4.0))
for st, col, lab in [(0, '#C44E52', 'Baseline (不分片)'),
                     (1, '#DD8452', 'ZeRO-1: 优化器状态分片'),
                     (2, '#55A868', 'ZeRO-2: +梯度分片'),
                     (3, '#4C72B0', 'ZeRO-3: +参数分片')]:
    if st == 0:
        ax.plot(N, np.full_like(N, B), color=col, lw=1.5, label=lab)
    else:
        ax.plot(N, [zero_mem(st, n) for n in N], color=col, lw=1.5, label=lab)
ax.set_xlabel('GPU 数量'); ax.set_ylabel('每卡显存 (GB)')
ax.set_title('7B 模型 ZeRO 分片：越到后期省得越多', fontsize=12)
ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print('8 卡 ZeRO-3: %.1fGB/卡 vs Baseline %.1fGB' % (zero_mem(3, 8), B))

### 实验 4 复盘：ZeRO 三阶段读图

- **Baseline**：每卡都要 84GB（横线）；
- **ZeRO-1**：优化器状态分片，8 卡时每卡 $84/8\cdot 8/12 + 28\approx 42$GB（省一半）；
- **ZeRO-2**：梯度也分片，8 卡 ≈ $70/8+14\approx 23$GB；
- **ZeRO-3**：参数也分片，8 卡 ≈ $84/8 = 10.5$GB——接近线性下降。

**代价是通信**：分片越彻底，all-gather/reduce-scatter 越频繁、通信量越大。
ZeRO-3 需要 NCCL 高速互联（NVLink/IB），否则卡间通信成为瓶颈。
> 面试记法：**ZeRO-1 优化器状态 / ZeRO-2 +梯度 / ZeRO-3 +参数**，通信量递增、内存递减。

## §3 显存换时间的组合拳

- **梯度累积**：等效大 batch，显存不变（见 05 篇）；
- **激活重计算（gradient checkpointing）**：不存中间激活，反向时重算 →
  激活显存从 $O(L)$ 降到 $O(\sqrt L)$ 量级，代价约 30% 训练时间；
- **ZeRO-offload**：优化器状态 / 梯度挪到 CPU 内存或 NVMe，GPU 显存进一步降；
- **流水并行（pipeline）+ 张量并行（tensor parallel）**：模型切分到多卡，
  配 ZeRO 组成「3D 并行」；数据并行负责 batch 维度。

> 🧮 一个 70B 模型要跑起来的最小配置：BF16 参数 140GB + 优化器状态（若 Adam）更多——
> 不 ZeRO 不并行根本装不下。这就是现代 LLM 训练框架（DeepSpeed / Megatron-LM / FSDP）存在的意义。

## §3.1 现代框架一句话

- **DeepSpeed ZeRO**：微软，优化器状态/梯度/参数分片 + offload，PyTorch 生态最常用；
- **FSDP（PyTorch）**：PyTorch 原生版 ZeRO-3，`sharding_strategy` 控制分片粒度；
- **Megatron-LM**：NVIDIA，张量并行 + 流水并行 + 序列并行，适合超大规模；
- **FlashAttention**：从注意力计算层面省显存/提速（与优化器无关但常被一起提）；
- **梯度累积/裁剪/EMA**：见 05 篇，LLM 训练标配。

> 面试若被问「70B 模型怎么训」，标准答案框架：BF16 混合精度 + AdamW + warmup/cosine
> + 梯度裁剪 + ZeRO-3/FSDP + 张量/流水并行 + 梯度累积 + 激活重计算。

## §4 面试速答

- **为什么 BF16 不是 FP16**：指数位 8=FP32，范围不溢出 → 免 loss scaling；尾数少但训练可容忍
- **7B Adam 状态口算**：$7\times10^9\times 12$B $= 84$GB
- **ZeRO 三阶段**：优化器状态 / +梯度 / +参数 依次分片，通信量递增
- **为什么优化器状态必须 FP32**：Adam 累加精度（小更新在 FP16 中被吞）
- **loss scaling 为什么退出历史**：BF16 范围与 FP32 相同，无需缩放
- **激活重计算换什么**：显存 $O(L)\to O(\sqrt L)$，时间 +30% 左右

## §5 自测清单

- [ ] 默写 FP16/BF16 指数位与最大范围（5/8 位；65504 / 3.4e38）
- [ ] 演示 loss scaling（本文实验 2）
- [ ] 口算 7B Adam 状态显存（84GB）
- [ ] 画出 ZeRO-1/2/3 内存曲线（本文已出图）
- [ ] 说清重计算/offload 换什么、花什么
- [ ] 简述 70B 模型训练的标准配置清单

> 💡 本笔记验收：`08-优化算法/README.md` 验收清单「混合精度 / ZeRO 内存计算」打勾。